# Modular latent operation (MLO): the random-target experiment

We edit the latent $z = \varphi(x)$ of a frozen text encoder through an operator on structured states, $\omega_Z(z) = \rho(z;\ s,\ \omega_S(s))$ with $s = \psi(z)$:

| piece | symbol | here |
|---|---|---|
| encoder | $\varphi$ | `Encoder("embeddinggemma", 256)`: the 256-wide prefix of the cached talent-motivation encodings |
| reader | $\psi(s \mid z)$ | `Reader(columns, hidden=2)`, cross-fitted over 2,000 pool units in 5 folds with `cross_fit` |
| operator | $\omega_S$ | any function `states DataFrame -> target states DataFrame`, here `LinearSCM(...).do(X=k)` per unit |
| renderer | $\rho$ | a `Renderer` subclass: writes $s'$ back into $z$ |

**Design**

- **States.** $\psi$ reads $S^+ = \{X, T, D, U, M\}$ once. $\rho$ is fitted per state version in `STATES`; on the main one, $S = \{X, T, D, U\}$, motivation `M` stays in the residual and is held constant in $Z$-space. Its preservation is read by the independent judge $\psi_{\mathrm{eval}}$, cross-fitted over the other 2,000 units (eval units included).
- **No operator in training.** $\rho$ sees observed $(z, s)$ pairs only. The minimal edit (`EditorRenderer`) learns from random targets $s' \sim \mathrm{Uniform}(S)$, so it can map any $z$ to any $s'$ and operators are swapped freely at test time.
- **Estimation gap.** Every $\rho$ is trained once on the true $s$ and once on the out-of-fold reading $\psi(z)$, and tested with both: $\psi$ = 5 folds × 5 seeds = 25 fits, $\rho$ = 6 kinds × 5 seeds × (2 arms on the main state version + 1 arm on each other one).
- **Wrong DAGs** are a test-time add-on (Section 7): same $\psi$ and $\rho$, another $\omega_S$.

The data name the sensitive attribute `X` (country-of-origin region; `O` in the paper). Fits and the evaluation are **cached** in `models/latent_lab/` (`latent_lab.cache.cached`): a cell reloads its result unless latent_lab's code, the cell's code or an input it reads changed.

Target latents $\varphi(x')$, the encodings of real texts with the target state, are used for evaluation only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from exp.sim_talent_motivation.talent_motivation import outcome_bayes_predictions, talent_posterior_accuracy
from latent_lab import (
    ConstantRenderer, EditorRenderer, Encoder, FlowRenderer, GroupOTRenderer, IdentityRenderer, LinearSCM,
    OTEditorRenderer, OTFlowRenderer, OutOfFold, Pipeline, Reader, StateEffectRenderer, cross_fit, evaluate, factual_score, mean_predict,
    mean_proba, state_metrics,
)
from latent_lab.cache import cached

ROOT = Path.cwd()  # the notebook sits in the repo root
SEEDS = range(5)

## 1. Data

The talent-motivation experiment (`exp/sim_talent_motivation/`): the talent SFM plus motivation `M`, a root and parent of `Y` only. Each evaluation unit has one target text, rendered with its state under its own $do(X = k)$ (`interventions.csv`).

`partition.csv` fixes the split of the 4,000 units:

| set | units | used for |
|---|---|---|
| **pool** | 2,800 in 5 folds of 560 | training $\psi$ (400 per fold), the judge (160 per fold) and $\rho$ (all) |
| **dev** | 200 | training the judge |
| **eval** | 1,000 | training the judge and evaluation; the only units with a target text $x'$ |

$\psi$ and the judge $\psi_{\mathrm{eval}}$ split the 4,000 units into two disjoint halves of 2,000, and each is cross-fitted over its half in 5 folds of 400 (so every fold model trains on 1,600 units):

- **$\psi$**: 400 seeded units of each pool fold. The $\psi$ of fold $k$ is fitted on the $\psi$ units of the other four folds and reads all 560 pool units of fold $k$, so every pool unit, also those of the judge, gets an out-of-fold reading $\psi(z)$.
- **The judge $\psi_{\mathrm{eval}}$**: the other 160 units of each pool fold (judge fold = pool fold), the dev units and the eval units (both spread over the 5 folds, seeded; `partition.csv` gives them `fold = -1`), i.e. 800 + 200 + 1,000 units, 40 dev and 200 eval per fold. The judge of fold $k$ reads the edited and target latents of the eval units in fold $k$ only, so it never reads a unit it was trained on.
- **$\rho$** trains on all 2,800 pool latents, with either their true states or their out-of-fold readings $\psi(z)$. It shares the 800 pool units of the judge (latents, and in the true-$s$ arm their states); $\psi$ and the judge share no unit.

The columns `L_rank` and `in_P` of `partition.csv` (the old 1,000-unit sets `L` for $\psi$ and `P` for the judge) are not used.

Note: use `df["T"]`, never `df.T` (that is pandas' transpose).

In [2]:
DATA = ROOT / "data/sim_talent_motivation"
factual = pd.read_csv(DATA / "sim_data_factual.csv", index_col="id")
counterfactual = pd.read_csv(DATA / "sim_data_counterfactual.csv", index_col="id")  # target states, under each unit's do(X = k)
do_value = pd.read_csv(DATA / "interventions.csv", index_col="id")["do_value"]
partition = pd.read_csv(DATA / "partition.csv", index_col="id")

LEVELS = {"X": 4, "T": 3, "D": 3, "U": 3, "M": 3}
FULL = list(LEVELS)            # what psi reads

STATES = {                     # what rho conditions on
    "S1": ["X", "T", "D", "U"],
    "S2": ["X", "D", "U"],
    "S3": ["X", "T", "D", "U", "M"],
    "S4": ["X"]
}
MAIN = "S1"                    # searched, trained on both true s and psi(z); the others on psi(z) only
STATE = STATES[MAIN]           # also the operator's SCM, the joint metrics and the search

pool = partition.index[partition["partition"] == "pool"]
eval_ids = partition.index[partition["partition"] == "eval"]
dev = partition.index[partition["partition"] == "dev"]
folds = partition.loc[pool, "fold"].to_numpy()   # psi's folds, over the pool

# psi: 400 units of each pool fold (2,000); the judge: the other 160 of each, dev and eval (2,000)
rng = np.random.default_rng(0)
psi_units = np.zeros(len(pool), bool)
for k in np.unique(folds):
    psi_units[rng.choice(np.flatnonzero(folds == k), 400, replace=False)] = True
judge_ids = pool[~psi_units].append(dev).append(eval_ids)
judge_folds = np.concatenate([folds[~psi_units], rng.permutation(len(dev)) % 5, rng.permutation(len(eval_ids)) % 5])
eval_folds = judge_folds[-len(eval_ids):]        # the judge's folds of the eval units, in eval_ids order

role = pd.Series(np.where(psi_units, "psi", "judge (pool)"), pool).reindex(judge_ids.append(pool[psi_units]))
role.loc[dev], role.loc[eval_ids] = "judge (dev)", "judge (eval)"
fold = pd.Series(np.concatenate([judge_folds, folds[psi_units]]), role.index)
pd.crosstab(role.rename("role"), fold.rename("fold"), margins=True)

fold,0,1,2,3,4,All
role,,,,,,
judge (dev),40,40,40,40,40,200
judge (eval),200,200,200,200,200,1000
judge (pool),160,160,160,160,160,800
psi,400,400,400,400,400,2000
All,800,800,800,800,800,4000


## 2. Encoder φ

`latents(experiment)` loads the cached encodings (no model download): `latents.factual(ids)` gives the source latents $\varphi(x)$, `latents.counterfactual(ids)` the target latents $\varphi(x')$ of the eval units.

On previous trials embeddinggemma with 768 dimensions had the best $\psi$-outputs, which is why we chose it. Smaller latent spaces and other models (Qwen and Nomic) were tried, too.

`encoders` holds its Matryoshka prefixes of width 128, 256, 512 and 768 (the first $d$ coordinates of the cached 768-wide encodings, renormalized). $\psi$ is compared across all of them (Section 3). Everything after $\psi$ uses one width, `ENCODER`: the 256-wide prefix, because at $d = 768$ the state flow's likelihood barely depends on $s$, so it hardly moves $z$.

In [3]:
encoders = {
    "embeddinggemma-128": Encoder("embeddinggemma", 128),
    "embeddinggemma-256": Encoder("embeddinggemma", 256),
    "embeddinggemma-512": Encoder("embeddinggemma", 512),
    "embeddinggemma-768": Encoder("embeddinggemma", 768)
}
for key, encoder in encoders.items():
    latents = encoder.latents("talent_motivation")
    encoders[key] = {
        "encoder": encoder,
        "latents": latents,
        "z_pool": latents.factual(pool),
        "s_pool": factual.loc[pool],
        "z_eval": latents.factual(eval_ids),
        "s_eval": factual.loc[eval_ids],
        "z_cf": latents.counterfactual(eval_ids),
        "s_cf": counterfactual.loc[eval_ids]
    }
    print(encoders[key]["z_pool"].shape, encoders[key]["z_eval"].shape)

# the width psi's readings, rho, the operator and the evaluation use from Section 3 on
ENCODER = "embeddinggemma-256"
encoder, latents, z_pool, s_pool, z_eval, s_eval, z_cf, s_cf = (
    encoders[ENCODER][k] for k in ("encoder", "latents", "z_pool", "s_pool", "z_eval", "s_eval", "z_cf", "s_cf")
)

torch.Size([2800, 128]) torch.Size([1000, 128])
torch.Size([2800, 256]) torch.Size([1000, 256])
torch.Size([2800, 512]) torch.Size([1000, 512])
torch.Size([2800, 768]) torch.Size([1000, 768])


## 3. Reader ψ, cross-fitted

`cross_fit` fits one reader per fold on the $\psi$ units of the other four pool folds (1,600) and reads the held-out fold, so every pool unit gets an **out-of-fold** reading $\psi(z)$ to train $\rho$ on. The eval units are read by the average of the five fold readers (`mean_predict`). Five seeds: 25 fits.

The judge $\psi_{\mathrm{eval}}$ has the same architecture, is cross-fitted the same way over its own 2,000 units (seed 0) and is used only to evaluate: `OutOfFold` reads every eval unit's latents, factual, edited or target, with the fold judge that was not trained on that unit. Its column in the table is therefore also an out-of-fold reading.

Both are fitted for every width in `encoders`, so the table compares how well each width can be read; `oof_joint_acc` is the joint accuracy over $S$ of the out-of-fold pool readings $\rho$ trains on. From here on only `ENCODER`'s $\psi$ and judge are used.

*Takes approx. 2 minutes per width*

In [4]:
# encoder -> seed -> (out-of-fold pool readings, fold readers); encoder -> judge
psi_by_encoder, judge_by_encoder = {}, {}
for key, e in encoders.items():
    z_w, s_w, z_j = e["z_pool"], e["s_pool"], e["latents"].factual(judge_ids)
    psi_by_encoder[key] = {seed: cached(f"psi-{key}-seed{seed}", lambda: cross_fit(Reader(FULL, hidden=2, levels=LEVELS),
                                                                                z_w, s_w, folds, psi_units, seed=seed))
                           for seed in tqdm(SEEDS, desc=f"psi {key} (5 folds per seed)")}
    _, judge_readers = cached(f"judge-{key}", lambda: cross_fit(Reader(FULL, hidden=2, levels=LEVELS),
                                                                z_j, factual.loc[judge_ids], judge_folds, seed=0))
    judge_by_encoder[key] = OutOfFold(judge_readers, eval_folds)   # reads eval units only, out of fold
psi, judge = psi_by_encoder[ENCODER], judge_by_encoder[ENCODER]   # the ones used from here on
s_read = {seed: mean_predict(readers, z_eval) for seed, (_, readers) in psi.items()}   # psi(z) on eval


def psi_summary(key: str) -> pd.DataFrame:
    # psi on eval (mean, sd over seeds) and the judge: per-variable accuracy, NLL and ECE; joint accuracy and NLL
    # over S; for psi also the joint accuracy over S of the out-of-fold pool readings rho trains on
    e, psi_w = encoders[key], psi_by_encoder[key]
    m = pd.concat([state_metrics(mean_proba(r, e["z_eval"]), e["s_eval"], joint=STATE) for _, r in psi_w.values()], axis=1)
    m.loc["oof_joint_acc"] = [(r[STATE].to_numpy() == e["s_pool"][STATE].to_numpy()).all(1).mean() for r, _ in psi_w.values()]
    return pd.DataFrame({
        "psi (mean)": m.mean(axis=1),
        "psi (sd over seeds)": m.std(axis=1),
        "judge": judge_by_encoder[key].metrics(e["z_eval"], e["s_eval"], joint=STATE),
    }).reindex(m.index)


CARD = {c: int(factual[c].max()) + 1 for c in factual}
print(f"Bayes ceiling of acc_T given X, D, U and the proxies: "
      f"{talent_posterior_accuracy(s_eval, ['X', 'D', 'U', 'P', 'L', 'H', 'A'], CARD):.3f}")
psi_table = pd.concat({key: psi_summary(key) for key in encoders}, axis=1)
psi_table.round(3)

psi embeddinggemma-768 (5 folds per seed): 100%|██████████| 5/5 [02:43<00:00, 32.69s/it]


Bayes ceiling of acc_T given X, D, U and the proxies: 0.936


embeddinggemma-128                             \
                      psi (mean) psi (sd over seeds)  judge   
acc_X                      0.995               0.001  0.992   
nll_X                      0.026               0.001  0.030   
ece_X                      0.012               0.001  0.012   
acc_T                      0.769               0.002  0.747   
nll_T                      0.559               0.003  0.601   
ece_T                      0.028               0.005  0.047   
acc_D                      0.992               0.001  0.982   
nll_D                      0.032               0.002  0.040   
ece_D                      0.014               0.001  0.010   
acc_U                      0.842               0.007  0.835   
nll_U                      0.383               0.007  0.390   
ece_U                      0.016               0.005  0.023   
acc_M                      0.603               0.008  0.588   
nll_M                      0.894               0.004  0.932   
ece_M                      0.025               0.006  0.035   
joint_acc                  0.650               0.008  0.615   
joint_nll                  1.000               0.010  1.060   
oof_joint_acc              0.628               0.006    NaN   

              embeddinggemma-256                             \
                      psi (mean) psi (sd over seeds)  judge   
acc_X                      0.995               0.001  0.994   
nll_X                      0.019               0.000  0.019   
ece_X                      0.009               0.001  0.011   
acc_T                      0.818               0.004  0.791   
nll_T                      0.443               0.004  0.520   
ece_T                      0.025               0.005  0.060   
acc_D                      0.999               0.001  0.997   
nll_D                      0.013               0.001  0.020   
ece_D                      0.011               0.001  0.011   
acc_U                      0.953               0.002  0.943   
nll_U                      0.138               0.004  0.156   
ece_U                      0.022               0.002  0.007   
acc_M                      0.688               0.005  0.643   
nll_M                      0.738               0.009  0.815   
ece_M                      0.026               0.009  0.048   
joint_acc                  0.778               0.006  0.743   
joint_nll                  0.613               0.004  0.715   
oof_joint_acc              0.748               0.007    NaN   

              embeddinggemma-512                             \
                      psi (mean) psi (sd over seeds)  judge   
acc_X                      0.995               0.001  0.994   
nll_X                      0.018               0.001  0.019   
ece_X                      0.008               0.001  0.009   
acc_T                      0.834               0.004  0.800   
nll_T                      0.396               0.004  0.491   
ece_T                      0.024               0.005  0.068   
acc_D                      0.999               0.001  0.999   
nll_D                      0.009               0.000  0.012   
ece_D                      0.008               0.000  0.008   
acc_U                      0.990               0.002  0.981   
nll_U                      0.055               0.003  0.064   
ece_U                      0.030               0.002  0.018   
acc_M                      0.766               0.004  0.730   
nll_M                      0.576               0.004  0.656   
ece_M                      0.026               0.011  0.036   
joint_acc                  0.822               0.005  0.780   
joint_nll                  0.478               0.005  0.585   
oof_joint_acc              0.793               0.004    NaN   

              embeddinggemma-768                             
                      psi (mean) psi (sd over seeds)  judge  
acc_X                      0.996               0.001  0.994  
nll_X                      0.015             

### Robustness of ψ to the wording

Does $\psi$ read the meaning of a level or the codebook's phrases? 500 eval units were also rendered with a held-out paraphrase of the codebook ($\mathcal V_B$; `z_factual_vb.pt`) next to the training wording ($\mathcal V_A$). Per variable: accuracy on the same units in both wordings and the gap $\mathcal V_A - \mathcal V_B$ with a 95 % paired bootstrap CI over units ($\psi$: averaged over seeds). `X` keeps its wording and serves as negative control. The judge is fitted on $\mathcal V_A$ too, so it is reported alongside: a large gap for it means $\psi_{\mathrm{eval}}$ needs $\mathcal V_B$ texts as well. One block per width in `encoders`.

In [5]:
vb = encoder.auxiliary("talent_motivation", "z_factual_vb.pt")   # the same units at every width
s_b = factual.loc[vb.ids]
vb_rows = eval_ids.get_indexer(vb.ids)   # their rows in eval_ids, for the out-of-fold judge
assert (vb_rows >= 0).all(), "the V_B units must be eval units"
boot = np.random.default_rng(0).integers(0, len(vb.ids), (2000, len(vb.ids)))   # paired: same units


def robustness(reads_a: list[pd.DataFrame], reads_b: list[pd.DataFrame]) -> pd.DataFrame:
    rows = {}
    for c in FULL:
        a = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_a], axis=0)   # per unit, over seeds
        b = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_b], axis=0)
        gaps = a[boot].mean(1) - b[boot].mean(1)
        rows[c] = {"V_A": a.mean(), "V_B": b.mean(), "gap": a.mean() - b.mean(),
                   "CI low": np.quantile(gaps, 0.025), "CI high": np.quantile(gaps, 0.975)}
    return pd.DataFrame(rows).T


def robustness_by_reader(key: str) -> pd.DataFrame:
    z_a = encoders[key]["latents"].factual(vb.ids)
    z_b = encoders[key]["encoder"].auxiliary("talent_motivation", "z_factual_vb.pt").factual(vb.ids)
    readers, judge_w = [r for _, r in psi_by_encoder[key].values()], judge_by_encoder[key].rows(vb_rows)
    return pd.concat({
        "psi": robustness([mean_predict(r, z_a) for r in readers], [mean_predict(r, z_b) for r in readers]),
        "judge": robustness([judge_w.predict(z_a)], [judge_w.predict(z_b)]),
    }, axis=1)


robustness_table = pd.concat({key: robustness_by_reader(key) for key in encoders})
robustness_table.round(3)

psi                               judge                \
                        V_A    V_B    gap CI low CI high    V_A    V_B    gap   
embeddinggemma-128 X  0.994  0.989  0.005 -0.005   0.016  0.990  0.990  0.000   
                   T  0.773  0.740  0.032 -0.004   0.069  0.738  0.700  0.038   
                   D  0.992  0.972  0.020  0.005   0.034  0.984  0.972  0.012   
                   U  0.854  0.765  0.089  0.048   0.130  0.844  0.754  0.090   
                   M  0.596  0.520  0.075  0.033   0.116  0.576  0.490  0.086   
embeddinggemma-256 X  0.994  0.992  0.002 -0.008   0.012  0.990  0.990  0.000   
                   T  0.824  0.802  0.022 -0.006   0.053  0.804  0.736  0.068   
                   D  0.999  0.998  0.002 -0.002   0.006  0.998  0.988  0.010   
                   U  0.962  0.850  0.111  0.079   0.144  0.950  0.830  0.120   
                   M  0.670  0.568  0.103  0.053   0.149  0.632  0.552  0.080   
embeddinggemma-512 X  0.994  0.992  0.002 -0.008   0.012  0.990  0.992 -0.002   
                   T  0.845  0.824  0.022 -0.008   0.051  0.818  0.774  0.044   
                   D  0.999  0.998  0.001 -0.001   0.004  0.998  0.998  0.000   
                   U  0.993  0.920  0.072  0.052   0.095  0.982  0.892  0.090   
                   M  0.772  0.672  0.100  0.056   0.143  0.730  0.636  0.094   
embeddinggemma-768 X  0.994  0.992  0.002 -0.007   0.012  0.992  0.992  0.000   
                   T  0.844  0.833  0.011 -0.018   0.042  0.816  0.776  0.040   
                   D  1.000  0.999  0.001  0.000   0.004  0.998  1.000 -0.002   
                   U  0.990  0.941  0.049  0.030   0.068  0.984  0.922  0.062   
                   M  0.794  0.695  0.099  0.057   0.140  0.740  0.656  0.084   

                                     
                     CI low CI high  
embeddinggemma-128 X -0.012   0.012  
                   T -0.004   0.080  
                   D -0.004   0.028  
                   U  0.046   0.134  
                   M  0.040   0.134  
embeddinggemma-256 X -0.012   0.012  
                   T  0.030   0.110  
                   D  0.000   0.020  
                   U  0.084   0.154  
                   M  0.030   0.130  
embeddinggemma-512 X -0.014   0.010  
                   T  0.008   0.082  
                   D -0.006   0.006  
                   U  0.064   0.118  
                   M  0.044   0.144  
embeddinggemma-768 X -0.012   0.012  
                   T  0.004   0.078  
                   D -0.006   0.000  
                   U  0.040   0.088  
                   M  0.034   0.134

## 4. Renderers ρ: 6 kinds × the state versions in `STATES` × 5 seeds, searched first

- `flow`: constant-noise edit, $u = F^{-1}(z; s)$, $z' = F(u; s')$.
- `editor`: minimal edit, trained on random targets $s' \sim \mathrm{Uniform}(S)$ against a critic fitted on the same training states.
- `ot_editor`: the same minimal edit with an input-convex gradient map, $x' = \nabla_x \phi(x; s')$.
- `ot_flow`: shared-reference OT flow, $u = F_s^{-1}(z)$, $z' = F_{s'}(u)$, fitted by OT flow matching within each state.
- `state_effect`, `group_ot`: closed-form baselines (additive state effect, Gaussian OT between the groups of `X`).

**State versions.** Each $\rho$ is fitted once per version in `STATES`, i.e. per set of columns it conditions on. $\psi$ and the operator stay the same: the operator acts on the whole reading, and $\rho$ only sees its columns of $s$ and $s'$. A column left out (e.g. `T` in S2, `D` and `U` in S4) stays in the residual; one added (`M` in S3) is held at $\psi$'s reading instead of in the residual. The main version `MAIN` is searched and trained on both true $s$ and $\psi(z)$; the others reuse its selected settings and are trained on $\psi(z)$ only, as in deployment.

`FlowRenderer` and `EditorRenderer` take under a minute each on CPU at $d = 256$; the whole loop runs for a while.

### Grid search, without target latents

Each configuration is fitted once (seed 0, true states) on pool folds 1–4 and scored on fold 0 with `factual_score`: every fold-0 unit is edited to the observed state of another, randomly drawn unit.

- `fidelity`: the share of units whose whole target state $\psi$'s fold-0 reader (fitted on folds 1–4, never on these units) reads in $z'$. Not the judge, which stays reserved for the evaluation.
- `realism`: for every column and level, the energy distance between edited latents with that target level and observed latents with that level, averaged. Lower is better.

Selection per kind: among configurations within 0.05 of the best fidelity, the lowest realism. Grids: learning rate × width for the two flows, proximity for the editor, proximity × learning rate for the OT editor (20 trials). The closed-form renderers are not searched. Target latents $\varphi(x')$ are not used, so the evaluation stays untouched. Every trial is cached.

**Blind spot.** Both criteria judge the edited latents as a *population*, never the individual unit. `realism` asks whether the edited latents with target level $k$ look like real latents with level $k$, not whether unit $i$ became *its own* version with level $k$. An edit that throws away what is specific to the unit (its persona, template and wording, and everything outside the conditioning set, such as `M` and the talent proxies) and returns a typical latent of the target state therefore scores well on both. In the extreme, replacing $z$ by a random real latent with state $s'$ scores perfectly. The search cannot tell such an edit from a good one. Only the evaluation can: `gap_closed` and `distance` compare each $z'$ with the same unit's target latent $\varphi(x')$, and the judge's `acc_M` and `acc_T` check what should have stayed unchanged. Check the selected configurations there. `shift` is reported but not used for selection, because a correct edit may need to move far.

In [ ]:
search_fit, search_val = folds != 0, folds == 0
z_sf, s_sf = z_pool[torch.as_tensor(search_fit)], s_pool[search_fit]
z_sv, s_sv = z_pool[torch.as_tensor(search_val)], s_pool[search_val]
selector = psi[0][1][0]   # psi's fold-0 reader: fitted on folds 1-4 only

GRID = {
    "flow": (FlowRenderer, [dict(epochs=500, lr=lr, hidden=h) for lr in (3e-4, 1e-3) for h in (68, 136)]),
    "ot_flow": (OTFlowRenderer, [dict(epochs=500, lr=lr, hidden=h) for lr in (3e-4, 1e-3) for h in (256, 512)]),
    "editor": (EditorRenderer, [dict(epochs=500, proximity=p) for p in (0.1, 0.3, 1.0, 3.0)]),
    # it learns slowly from its near-identity start, so its learning rate is searched too
    "ot_editor": (OTEditorRenderer, [dict(epochs=500, lr=lr, proximity=p) for lr in (1e-3, 1e-2) for p in (0.1, 0.3, 1.0, 3.0)]),
}
trials = [(kind, cls, config) for kind, (cls, configs) in GRID.items() for config in configs]
rows, search_fits = [], {}
for kind, cls, config in (bar := tqdm(trials, desc="search")):
    bar.set_postfix(kind=kind, **config)
    name = f"search-{kind}-" + "-".join(f"{k}{v}" for k, v in config.items())
    trial = cached(name, lambda: cls(STATE, levels=LEVELS, **config).fit(z_sf, s_sf, seed=0))
    search_fits[(kind, str(config))] = trial
    rows.append({"kind": kind, "config": config, **factual_score(trial, z_sv, s_sv, selector)})
search = pd.DataFrame(rows)


def select(trials: pd.DataFrame) -> dict:
    eligible = trials[trials["fidelity"] >= trials["fidelity"].max() - 0.05]
    return eligible.loc[eligible["realism"].idxmin(), "config"]


best = {kind: select(trials) for kind, trials in search.groupby("kind")}
print("selected:", best)
search.assign(config=search["config"].map(str)).set_index(["kind", "config"]).round(3)

In [ ]:
# searched kinds use the selected configuration; the default argument keeps each one's
# cache key to its own settings
RHOS = {
    "flow": lambda columns, c=best["flow"]: FlowRenderer(columns, levels=LEVELS, **c),
    "editor": lambda columns, c=best["editor"]: EditorRenderer(columns, levels=LEVELS, **c),
    "ot_editor": lambda columns, c=best["ot_editor"]: OTEditorRenderer(columns, levels=LEVELS, **c),
    "ot_flow": lambda columns, c=best["ot_flow"]: OTFlowRenderer(columns, levels=LEVELS, **c),
    "state_effect": lambda columns: StateEffectRenderer(columns, levels=LEVELS),
    "group_ot": lambda columns: GroupOTRenderer(columns, on="X", shrink=0.1, levels=LEVELS),
}
runs = [(S, kind, arm, seed) for S in STATES for seed in SEEDS
        for arm in (("true s", "ψ(z)") if S == MAIN else ("ψ(z)",)) for kind in RHOS]
rho = {}   # (state version, kind, train states, seed) -> fitted renderer
for S, kind, arm, seed in (bar := tqdm(runs, desc="rho")):
    bar.set_postfix(S=S, kind=kind, train=arm, seed=seed)
    states, columns = (s_pool if arm == "true s" else psi[seed][0]), STATES[S]
    make = RHOS[kind]  # only this renderer's settings enter the cache key, not all of RHOS
    name = f"rho-{S}-{kind}-{'true' if arm == 'true s' else 'psi'}-seed{seed}"
    rho[(S, kind, arm, seed)] = cached(name, lambda: make(columns).fit(z_pool, states, seed=seed))
len(rho)

## 5. Operator $\omega_S$

$\rho$ was fitted without any operator; $\omega_S$ enters only now, between reading and rendering. Each eval unit's $do(X = k)$ on the SCM over $S$, with the numbers copied from `exp/sim_talent/talent_sfm.py` (**editing them changes only the operator, not the data**). Columns outside the SCM, such as `M`, pass through unchanged. `operator(seed)` builds a freshly seeded SCM, so every run sees the same abduction draws.

The SCM operator follows abduction (noise drawn from the range consistent with the observed state), action (set the do() values) and prediction (recompute the downstream nodes); `blocked={"D": ["X"]}` makes `D` read the observed `X`.

In [ ]:
PARENTS = {"D": {"X": 0.30, "T": 0.60}, "U": {"X": 0.35, "T": 0.50}}   # node: {parent: weight}
NOISE = {"D": (-0.10, 0.50), "U": (-0.15, 0.50)}                        # node: (mu, sd)
do_eval = do_value.loc[eval_ids].to_numpy()


def operator(seed=0, parents=PARENTS, blocked=None):
    scm = LinearSCM({c: LEVELS[c] for c in STATE}, parents, NOISE, seed=seed)
    return scm.do(X=lambda s: do_eval, blocked=blocked)


print(LinearSCM({c: LEVELS[c] for c in STATE}, PARENTS, NOISE))
# sanity check: the operator on the true source states vs. the simulated target states
pd.DataFrame({
    "operator": operator()(s_eval[STATE].reset_index(drop=True))["U"].value_counts(normalize=True),
    "simulated": s_cf["U"].value_counts(normalize=True),
}).sort_index().round(3)

## 6. Downstream predictor π

A frozen MLP $\pi: Z \to Y$ (a one-column `Reader`), fitted on the observed pool latents and their outcome `Y`, with the epoch selected on a held-out 20 % of the pool. It never sees target data. References: $\pi$ on the real $\varphi(x')$, and the Bayes-optimal accuracy for $Y'$ given the true counterfactual state with and without `M`.

`score(z')` collects every metric of one edit, against the true target.

In [ ]:
pi = cached("pi", lambda: Reader(["Y"], hidden=2, levels={"Y": CARD["Y"]}).fit(z_pool, s_pool, seed=0))


def predict(z: torch.Tensor) -> torch.Tensor:
    return pi.proba(z)["Y"].argmax(-1)


def score(z_prime: torch.Tensor) -> pd.Series:
    return evaluate(z_eval, z_prime, z_cf, judge, s_cf, joint=STATE,
                    predict=predict, outcome=s_cf["Y"], groups=s_cf["X"])


y_cf = s_cf["Y"].to_numpy()
outcome_table = pd.Series({
    "pi(z) vs factual Y": float((predict(z_eval).numpy() == s_eval["Y"].to_numpy()).mean()),
    "pi(phi(x')) vs Y'": float((predict(z_cf).numpy() == y_cf).mean()),
    **{f"Bayes vs Y' | {name}": float((p.argmax(1) == y_cf).mean())
       for name, p in outcome_bayes_predictions(s_cf).items()},
})
outcome_table.round(3)

## 7. Evaluation under each unit's do(X = k)

Every $\rho$ is tested with the true source states and with $\psi(z)$, per state version `S`. Train × test gives the estimation gap: the `true s / true s` cell is the privileged upper bound, `ψ(z) / ψ(z)` is deployment.

- **Fidelity and preservation:** the judge's `joint_acc` over $S$ and `acc_*` on $z'$ against the true target states; `acc_M` and `acc_T` are preservation (neither changes under $do(X)$, and $\rho$ never sees `M`). `real_*` is the judge on $\varphi(x')$ itself: reading $z'$ *better* than a real latent points to off-manifold edits.
- **Latent:** `gap_closed` to the paired $\varphi(x')$ (1 = perfect, 0 = no better than $z$), `distance` $|z' - \varphi(x')|$ (the energy score of a point edit), `energy` distance between the sets $\{z'\}$ and $\{\varphi(x')\}$, `shift` $|z' - z|$; all in standardized coordinates, divided by $\sqrt d$.
- **Downstream:** `down_acc` of $\pi(z')$ against $Y'$, `down_agree` with $\pi(z)$, and `parity_gap`, the range over the target regions of the mean predicted $Y$.

In [ ]:
def evaluate_all() -> pd.DataFrame:
    rows = []
    for (S, kind, arm, seed), renderer in tqdm(rho.items(), desc="evaluate", total=len(rho)):
        for test, states in {"true s": s_eval, "ψ(z)": s_read[seed]}.items():
            result = Pipeline(None, operator(seed), renderer)(z_eval, states)
            rows.append({"S": S, "rho": kind, "train": arm, "test": test, "seed": seed, **score(result.z_prime)})
    for kind, renderer in {"identity": IdentityRenderer(STATE),
                           "constant": ConstantRenderer(STATE, on="X").fit(z_pool, s_pool)}.items():
        result = Pipeline(None, operator(), renderer)(z_eval, s_eval)
        rows.append({"S": MAIN, "rho": kind, "train": "-", "test": "true s", "seed": 0, **score(result.z_prime)})
    return pd.DataFrame(rows)


results = cached("results", evaluate_all)

metrics = ["joint_acc", "acc_X", "acc_D", "acc_U", "acc_T", "acc_M",
           "gap_closed", "distance", "energy", "shift", "down_acc", "down_agree", "parity_gap"]
print("references on the real phi(x'):", results.filter(like="real_").iloc[0].round(3).to_dict())
summary = results.groupby(["S", "rho", "train", "test"])[metrics].agg(["mean", "std"])
summary.xs("mean", axis=1, level=1).round(3)

In [ ]:
# estimation gap per renderer (main state version): train states x test states
main = results[(results["S"] == MAIN) & (results["train"] != "-")]
main.pivot_table(["joint_acc", "gap_closed", "down_acc"], "rho", ["train", "test"]).round(3)

In [ ]:
# state versions side by side: trained and tested on psi(z), mean over seeds
deploy = results[(results["train"] == "ψ(z)") & (results["test"] == "ψ(z)")]
deploy.pivot_table(["gap_closed", "acc_X", "acc_D", "acc_U", "acc_T", "acc_M"], "rho", "S").round(3)

## 8. Add-on: wrong DAGs at test time

$\psi$ and $\rho$ stay fixed; only $\omega_S$ changes. Shown for the flow on the main state version, trained and tested on $\psi(z)$, seed 0, against the true target.

In [ ]:
wrong = {
    "true SCM": operator(),
    "direct effect (D, U blocked)": operator(blocked={"D": ["X"], "U": ["X"]}),
    "wrong SCM (no X->D)": operator(parents={**PARENTS, "D": {"T": 0.60}}),
}
renderer = rho[(MAIN, "flow", "ψ(z)", 0)]
wrong_table = pd.DataFrame({
    name: score(Pipeline(None, op, renderer)(z_eval, s_read[0]).z_prime)
    for name, op in wrong.items()
}).T
wrong_table[metrics].round(3)

## 9. Store the run

One JSON file per run in `reports/latent_lab/`: the settings, the code version (git commit, and whether `latent_lab/` had uncommitted changes) every table above at full precision, and the training time and convergence of every fit (seconds, epochs run, the epoch kept, early stopping, and the loss per epoch). Times are those of the original fit, also for models reloaded from the cache. The fitted models stay in the cache (`models/latent_lab/`).

In [ ]:
import json
import subprocess
from datetime import datetime


def table(frame) -> list | dict:
    # DataFrames as one record per row, index included; Series as a dict
    if isinstance(frame, pd.Series):
        return {str(k): v for k, v in frame.items()}
    frame = frame.copy()
    frame.columns = [" | ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in frame.columns]
    frame = frame if isinstance(frame.index, pd.RangeIndex) else frame.reset_index()
    return frame.to_dict(orient="records")


def git(*args: str) -> str:
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()


run = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "latent_lab_uncommitted_changes": bool(git("status", "--porcelain", "latent_lab")),
    "settings": {
        "encoder": repr(encoder), "psi_encoders": {k: repr(e["encoder"]) for k, e in encoders.items()},
        "seeds": list(SEEDS), "states": STATES, "main": MAIN, "read": FULL, "levels": LEVELS,
        "units": {"pool": len(pool), "psi": int(psi_units.sum()), "judge": len(judge_ids), "eval": len(eval_ids)},
        "search_grid": {kind: configs for kind, (_, configs) in GRID.items()},
        "selected": best,
        "renderers": {kind: {"class": type(r).__name__, **{k: v for k, v in vars(r).items() if not k.startswith("_")}}
                      for kind, r in ((kind, make(STATE)) for kind, make in RHOS.items())},
    },
    "psi": table(psi_table),
    "robustness": table(robustness_table),
    "search": table(search.assign(config=search["config"].map(str))),
    "outcome": table(outcome_table),
    "results": table(results),
    "summary": table(results.groupby(["S", "rho", "train", "test"])[metrics].agg(["mean", "std"])),
    "wrong_dag": table(wrong_table),
}
# training time and convergence of every fit (as originally trained, also when reloaded from the cache)
fits = (
    [{"model": "psi", "encoder": key, "seed": seed, "fold": fold, **r.fit_log}
     for key, psi_w in psi_by_encoder.items() for seed, (_, readers) in psi_w.items() for fold, r in enumerate(readers)]
    + [{"model": "judge", "encoder": key, "fold": fold, **r.fit_log}
       for key, j in judge_by_encoder.items() for fold, r in enumerate(j.readers)]
    + [{"model": "pi", **pi.fit_log}]
    + [{"model": f"search-{kind}", "config": config, **t.fit_log} for (kind, config), t in search_fits.items()]
    + [{"model": f"rho-{kind}", "S": S, "train": arm, "seed": seed, **r.fit_log} for (S, kind, arm, seed), r in rho.items()]
)
run["training"] = fits
path = ROOT / "reports" / "latent_lab" / f"run-{datetime.now():%Y%m%d-%H%M%S}.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(run, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("stored", path)
training_table = pd.DataFrame([{k: v for k, v in f.items() if k != "history"} for f in fits])
print(f"total training time: {training_table['seconds'].sum() / 60:.1f} min")
training_table.groupby("model")[["seconds", "epochs_run", "best_epoch"]].agg(["mean", "max"]).round(1)